# Nemotron Colab Experiment Hub

这个 notebook 用于在 Colab 快速跑通并对比多种策略。

支持的方法：
- `plain`: 直接问答
- `typed_hint`: 按题型附加简短规则提示
- `fewshot`: 统一 few-shot 示例
- `self_consistency`: 同一题多次采样后多数投票

支持的后端：
- `hf`: Hugging Face 本地推理（Colab 默认先用小模型验证流程）
- `openai_compat`: OpenAI 兼容 API（可接 vLLM 服务或网关）

> 说明：竞赛正式评测用 vLLM + Nemotron + LoRA。这个 notebook 优先解决“Colab 上快速实验各种方法”的问题。

In [ ]:
%pip -q install -U pandas numpy tqdm transformers accelerate peft openai

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import os
import re
import json
import random

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

METRIC_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

BOX_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
BOX_OPEN_RE = re.compile(r"\\boxed\{([^}]*)$")

def infer_type(prompt: str) -> str:
    p = prompt.lower()
    if "binary" in p or "bit" in p:
        return "bit_ops"
    if "cipher" in p or "encrypt" in p or "decrypt" in p:
        return "cipher"
    if "gravity" in p or "fall" in p or "m/s" in p:
        return "gravity"
    if "roman" in p or "numeral" in p:
        return "numeral"
    if "convert" in p or "unit" in p:
        return "unit_conv"
    if "symbol" in p or "equation" in p:
        return "symbol"
    return "unknown"

def extract_boxed(text: str) -> str:
    if not text:
        return "NOT_FOUND"
    matches = BOX_RE.findall(text)
    if matches:
        val = matches[-1].strip()
        return val if val else "NOT_FOUND"
    m = BOX_OPEN_RE.search(text.strip())
    if m:
        val = m.group(1).strip()
        return val if val else "NOT_FOUND"
    return "NOT_FOUND"

def normalize_answer(x: str) -> str:
    return str(x).strip()

def is_correct(pred: str, gold: str, tol: float = 1e-2) -> bool:
    p = normalize_answer(pred)
    g = normalize_answer(gold)
    if p == g:
        return True
    try:
        return abs(float(p) - float(g)) <= tol
    except Exception:
        return False

In [ ]:
# Colab 可选挂载
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # 例子：DATA_ROOT = '/content/drive/MyDrive/Nemotron'

DATA_ROOT = '/content'
TRAIN_CSV = os.path.join(DATA_ROOT, 'competition_data', 'train.csv')
TEST_CSV = os.path.join(DATA_ROOT, 'competition_data', 'test.csv')

print('TRAIN_CSV =', TRAIN_CSV)
print('TEST_CSV  =', TEST_CSV)
print('如果路径不存在，请先把项目目录上传到 /content 或改 DATA_ROOT。')

In [ ]:
@dataclass
class RunConfig:
    backend: str = 'hf'  # 'hf' or 'openai_compat'
    model_name_or_path: str = 'Qwen/Qwen2.5-3B-Instruct'
    adapter_path: Optional[str] = None

    # openai-compatible API settings
    api_base: Optional[str] = None
    api_key_env: str = 'OPENAI_API_KEY'

    temperature: float = 0.0
    top_p: float = 1.0
    max_new_tokens: int = 768

    eval_limit: int = 120
    sample_seed: int = 42

    self_consistency_k: int = 3

CFG = RunConfig()
CFG

In [ ]:
RULE_HINTS = {
    'bit_ops': 'Think per bit position and infer the boolean rule from examples.',
    'cipher': 'Infer a consistent bijection/mapping from examples, then decode or encode carefully.',
    'gravity': 'Track unit consistency and compute using the implied gravity relation from examples.',
    'numeral': 'Infer the numeral symbol system then convert exactly.',
    'unit_conv': 'Infer conversion factors from examples before applying to target value.',
    'symbol': 'Infer transformation rule pattern from all examples before solving target equation.',
    'unknown': 'Infer the hidden rule from examples step by step.'
}

FEWSHOT_BLOCK = '''
Example style:
Q: Given examples, infer rule and answer.
A: I infer the mapping/rule from all pairs, apply to target, and return only final answer as \boxed{...}.
'''

def build_prompt(prompt: str, method: str) -> str:
    t = infer_type(prompt)
    if method == 'plain':
        return prompt + METRIC_SUFFIX
    if method == 'typed_hint':
        hint = RULE_HINTS.get(t, RULE_HINTS['unknown'])
        return f"{prompt}\n\nHint: {hint}{METRIC_SUFFIX}"
    if method == 'fewshot':
        return f"{FEWSHOT_BLOCK}\n\nTask:\n{prompt}{METRIC_SUFFIX}"
    return prompt + METRIC_SUFFIX

In [ ]:
class HFBackend:
    def __init__(self, cfg: RunConfig):
        self.cfg = cfg
        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.model_name_or_path, use_fast=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            cfg.model_name_or_path,
            torch_dtype=dtype,
            device_map='auto'
        )
        if cfg.adapter_path:
            self.model = PeftModel.from_pretrained(self.model, cfg.adapter_path)
            self.model = self.model.merge_and_unload()

    def generate_once(self, user_prompt: str) -> str:
        messages = [{"role": "user", "content": user_prompt}]
        chat = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = self.tokenizer(chat, return_tensors='pt').to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                do_sample=self.cfg.temperature > 0,
                temperature=max(self.cfg.temperature, 1e-5),
                top_p=self.cfg.top_p,
                max_new_tokens=self.cfg.max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id
            )
        text = self.tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return text

class OpenAICompatBackend:
    def __init__(self, cfg: RunConfig):
        from openai import OpenAI

        key = os.environ.get(cfg.api_key_env, '')
        if not cfg.api_base:
            raise ValueError('api_base 不能为空，例如 http://<host>:8000/v1')
        self.client = OpenAI(base_url=cfg.api_base, api_key=key if key else 'EMPTY')
        self.cfg = cfg

    def generate_once(self, user_prompt: str) -> str:
        r = self.client.chat.completions.create(
            model=self.cfg.model_name_or_path,
            messages=[{'role': 'user', 'content': user_prompt}],
            temperature=self.cfg.temperature,
            top_p=self.cfg.top_p,
            max_tokens=self.cfg.max_new_tokens
        )
        return r.choices[0].message.content or ''

def make_backend(cfg: RunConfig):
    if cfg.backend == 'hf':
        return HFBackend(cfg)
    if cfg.backend == 'openai_compat':
        return OpenAICompatBackend(cfg)
    raise ValueError(f'未知 backend: {cfg.backend}')

In [ ]:
def majority_vote(items: List[str]) -> str:
    cnt: Dict[str, int] = {}
    for x in items:
        k = normalize_answer(x)
        cnt[k] = cnt.get(k, 0) + 1
    return sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[0][0] if cnt else 'NOT_FOUND'

def evaluate_methods(
    df: pd.DataFrame,
    backend,
    methods: List[str],
    cfg: RunConfig
) -> pd.DataFrame:
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        prompt = str(row['prompt'])
        gold = str(row['answer']) if 'answer' in row else None
        qid = row['id']
        qtype = infer_type(prompt)

        for m in methods:
            built = build_prompt(prompt, m if m != 'self_consistency' else 'plain')

            if m == 'self_consistency':
                preds = []
                for _ in range(cfg.self_consistency_k):
                    raw = backend.generate_once(built)
                    preds.append(extract_boxed(raw))
                pred = majority_vote(preds)
                raw_text = '\n---\n'.join(preds)
            else:
                raw = backend.generate_once(built)
                pred = extract_boxed(raw)
                raw_text = raw

            rec = {
                'id': qid,
                'type': qtype,
                'method': m,
                'prediction': pred,
                'raw_output': raw_text,
            }
            if gold is not None:
                rec['gold'] = gold
                rec['correct'] = is_correct(pred, gold)
            records.append(rec)

    return pd.DataFrame(records)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)

if CFG.eval_limit > 0 and CFG.eval_limit < len(train_df):
    eval_df = train_df.sample(n=CFG.eval_limit, random_state=CFG.sample_seed).reset_index(drop=True)
else:
    eval_df = train_df.copy()

print('eval_df size =', len(eval_df))
eval_df.head(2)

In [ ]:
METHODS = ['plain', 'typed_hint', 'fewshot', 'self_consistency']

backend = make_backend(CFG)
results = evaluate_methods(eval_df, backend, METHODS, CFG)

results.head()

In [ ]:
summary = (
    results.groupby(['method'])['correct']
    .mean()
    .sort_values(ascending=False)
    .rename('acc')
    .reset_index()
)
display(summary)

summary_by_type = (
    results.groupby(['method', 'type'])['correct']
    .mean()
    .rename('acc')
    .reset_index()
    .sort_values(['method', 'acc'], ascending=[True, False])
)
display(summary_by_type)

In [ ]:
OUT_DIR = '/content/outputs'
os.makedirs(OUT_DIR, exist_ok=True)

results_path = os.path.join(OUT_DIR, 'method_compare_results.csv')
summary_path = os.path.join(OUT_DIR, 'method_compare_summary.csv')
cfg_path = os.path.join(OUT_DIR, 'run_config.json')

results.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)
with open(cfg_path, 'w', encoding='utf-8') as f:
    json.dump(CFG.__dict__, f, ensure_ascii=False, indent=2)

print('Saved:', results_path)
print('Saved:', summary_path)
print('Saved:', cfg_path)

## 下一步建议

1. 先用 `hf + 小模型` 验证流程正确（5-20 样本）。
2. 再切到你自己的 `openai_compat` / vLLM 服务，复现实验口径。
3. 如果要做 LoRA 对比：固定同一批 eval 样本，只改 `adapter_path` 做 A/B。
4. 如果要贴近官方：把 `temperature=0.0`，并统一 `METRIC_SUFFIX`。